# EGX daily LSTM - one stock, one month train / one month test

Prepares the csv files written by `pull_niph.py` / `pull_egx_stocks.py` into LSTM
windows, trains one model **per stock**, and scores it.

**The setup**

| | |
|---|---|
| feature per day | `(close - open) / open * 100`, and volume normalised by the **month's** max volume |
| window | the last `SEQ_LEN = 3` days -> an LSTM over 3 timesteps |
| target | the **next** day's `(close - open) %` |
| train / test | month 8 / month 9 of the same year |
| regression score | RMSE, against a predict-nothing baseline |
| classification | `>= +1.5%` up, `<= -1.5%` down, in between tie |
| headline metric | **precision of "up"** - a false positive is the model saying up when the day ties or falls |

The model regresses the next day's percentage; the up/tie/down classes are read off
that number. So one model gives you both the RMSE and the confusion matrix.

> **Read this before trusting any number below.** One EGX month is about 20 trading
> days, so month 8 yields roughly **17 training windows** and month 9 about the same
> for test. That is three orders of magnitude less data than an LSTM needs - whatever
> precision comes out is a property of those ~17 rows, not of the stock. Treat the
> first run as a pipeline check. To make it a real experiment, widen `TRAIN_MONTHS`
> (it is a list: `[1, 2, 3, 4, 5, 6, 7]`) and keep month 9 as the held-out test.

## 1. Config

Everything you would want to change lives here. **Change `STOCK` to train a different
stock** - the csv file name is derived from it, matching what the scraper scripts write.

In [ ]:
# ---- what to train on -------------------------------------------------------
STOCK = "NIPH"          # <-- change this to train another stock
EXCHANGE = "EGX"
YEAR = None             # None -> the most recent year that has both months

TRAIN_MONTHS = [8]      # a list, so it widens to [1,2,3,4,5,6,7] without other edits
TEST_MONTHS = [9]

# ---- how the windows are built ----------------------------------------------
SEQ_LEN = 3             # days of history per sample -> LSTM timesteps
VOLUME_NORM = "month"   # "month" = each month by its own max (as specified)
                        # "train" = every month by the training max (no lookahead)

# ---- the up / tie / down rule ------------------------------------------------
UP_THRESHOLD = 1.5      # >= +1.5%  -> up
DOWN_THRESHOLD = -1.5   # <= -1.5%  -> down, anything between the two is a tie

# The class the *prediction* is read off with. Keep it equal to UP_THRESHOLD for the
# rule as written; raise it to buy precision at the cost of recall (cell 8 sweeps it).
UP_DECISION = UP_THRESHOLD
DOWN_DECISION = DOWN_THRESHOLD

# ---- the model ---------------------------------------------------------------
LSTM_UNITS = 8          # keep this small, the training set is tiny
EPOCHS = 300
BATCH_SIZE = 8
LEARNING_RATE = 1e-3
SEED = 42

# ---- where the csv files are -------------------------------------------------
DATA_DIR = "exported_files"
CSV_NAME = "{exchange}_{stock}_1D.csv"   # what pull_egx_stocks.py writes

## 2. Imports

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam

def seed_everything(seed=SEED):
    """same seed for python, numpy and tensorflow, so a rerun repeats"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything()
print("tensorflow", tf.__version__)

## 3. Find the csv

Looks next to the notebook, then in Drive, and falls back to an upload prompt. The
file is the one the scraper writes, e.g. `EGX_NIPH_1D.csv`.

In [ ]:
def csv_path(stock, exchange=EXCHANGE, data_dir=DATA_DIR):
    """locate <EXCHANGE>_<STOCK>_1D.csv, asking for an upload as a last resort"""
    name = CSV_NAME.format(exchange=exchange, stock=stock)

    candidates = [
        os.path.join(data_dir, name),
        os.path.join("/content", data_dir, name),
        os.path.join("/content", name),
        os.path.join("/content/drive/MyDrive", data_dir, name),
        os.path.join("/content/drive/MyDrive", name),
    ]
    for path in candidates:
        if os.path.exists(path):
            print(f"using {path}")
            return path

    # nothing on disk, ask colab for it
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"{name} not found. Looked in:\n  " + "\n  ".join(candidates)
        )

    print(f"{name} not found, please upload it")
    uploaded = files.upload()
    if name in uploaded:
        return name
    if uploaded:
        return list(uploaded)[0]
    raise FileNotFoundError(f"{name} was not uploaded")


def mount_drive():
    """optional, run this if your csv files live in Google Drive"""
    from google.colab import drive
    drive.mount("/content/drive")

## 4. Load and prepare

Two columns come out of this: `pct` = `(close - open) / open * 100`, and
`volume_norm` = volume over the max volume of its month.

`VOLUME_NORM = "month"` is the rule as specified. Worth knowing what it means: the
test month is scaled by its own maximum, which is a number you would not have known
on the day. It does not leak the *target* (that is `pct`, untouched), but it does
leak the month's volume envelope. `VOLUME_NORM = "train"` scales everything by the
training max instead, which is what a live model would have to do.

In [ ]:
def load_prices(path):
    """read the scraper's csv into one row per trading day"""
    df = pd.read_csv(path)

    date_col = "datetime" if "datetime" in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.rename(columns={date_col: "datetime"})

    df = df.sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

    # an open of 0 would blow up the percentage
    bad = df["open"] <= 0
    if bad.any():
        print(f"dropping {int(bad.sum())} rows with a non positive open")
        df = df[~bad].reset_index(drop=True)

    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["pct"] = (df["close"] - df["open"]) / df["open"] * 100.0
    return df


def pick_year(df, year=YEAR, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS):
    """the requested year, or the most recent one holding every month we need"""
    needed = set(train_months) | set(test_months)
    if year is not None:
        return year

    have = df.groupby("year")["month"].apply(lambda m: needed <= set(m))
    years = [int(y) for y, ok in have.items() if ok]
    if not years:
        raise ValueError(
            f"no year in the data has all of months {sorted(needed)}. "
            f"Months per year:\n{df.groupby('year')['month'].unique()}"
        )
    return max(years)


def add_volume_norm(df, mode=VOLUME_NORM, train_months=TRAIN_MONTHS):
    """volume scaled onto 0..1, by month or by the training months"""
    df = df.copy()
    if mode == "month":
        peak = df.groupby(["year", "month"])["volume"].transform("max")
    elif mode == "train":
        peak = df.loc[df["month"].isin(train_months), "volume"].max()
    else:
        raise ValueError(f"unknown VOLUME_NORM {mode!r}, use 'month' or 'train'")

    peak = pd.Series(peak, index=df.index).replace(0, np.nan)
    df["volume_norm"] = (df["volume"] / peak).fillna(0.0)
    return df

## 5. Windows

Each sample is `SEQ_LEN` consecutive days of `[pct, volume_norm]`, and the label is
the **next** day's `pct`.

Windows never straddle a month boundary. That is what keeps the split honest: the
last days of August would otherwise be predicting September days that belong to the
test set.

In [ ]:
def make_windows(df, months, seq_len=SEQ_LEN):
    """(n, seq_len, 2) features, (n,) targets, and the date each target falls on"""
    X, y, dates = [], [], []

    for month in months:
        block = df[df["month"] == month].sort_values("datetime")
        if len(block) <= seq_len:
            print(f"month {month}: {len(block)} rows, too few for a {seq_len} day window")
            continue

        feats = block[["pct", "volume_norm"]].to_numpy(dtype="float32")
        target = block["pct"].to_numpy(dtype="float32")
        when = block["datetime"].to_numpy()

        # day i is predicted from the seq_len days before it, all inside this month
        for i in range(seq_len, len(block)):
            X.append(feats[i - seq_len:i])
            y.append(target[i])
            dates.append(when[i])

    if not X:
        raise ValueError(f"no windows built for months {months}")

    return np.stack(X), np.array(y, dtype="float32"), np.array(dates)


def classify(pct, up=UP_THRESHOLD, down=DOWN_THRESHOLD):
    """the up / tie / down rule, vectorised"""
    pct = np.asarray(pct, dtype="float64")
    return np.where(pct >= up, "up", np.where(pct <= down, "down", "tie"))

## 6. The model

An LSTM reading 3 timesteps of 2 features, then one linear output - the next day's
percentage. `LSTM_UNITS` is the width of the hidden state, not the 3 timesteps; those
come from the shape of the input.

In [ ]:
def build_model(seq_len=SEQ_LEN, n_features=2, units=LSTM_UNITS, lr=LEARNING_RATE):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(units),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse")
    return model


def train_model(model, X, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0):
    history = model.fit(
        X, y, epochs=epochs, batch_size=batch_size, shuffle=False, verbose=verbose
    )
    return history

## 7. Scoring

RMSE on its own says little, so it is printed next to two baselines: always predicting
0%, and always predicting the training mean. A model that cannot beat those has learnt
nothing.

Then the confusion matrix, and the number this is really about: **precision of "up"**.
A false positive is the model calling up on a day that ties or falls.

In [ ]:
CLASSES = ["down", "tie", "up"]


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def confusion(y_true_cls, y_pred_cls, classes=CLASSES):
    """rows are the truth, columns are the prediction"""
    idx = {c: i for i, c in enumerate(classes)}
    m = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true_cls, y_pred_cls):
        m[idx[t], idx[p]] += 1
    return m


def per_class_scores(cm, classes=CLASSES):
    """precision / recall / support per class, straight off the matrix"""
    rows = []
    for i, c in enumerate(classes):
        tp = cm[i, i]
        predicted = cm[:, i].sum()
        actual = cm[i, :].sum()
        rows.append({
            "class": c,
            "precision": tp / predicted if predicted else np.nan,
            "recall": tp / actual if actual else np.nan,
            "predicted": int(predicted),
            "support": int(actual),
        })
    return pd.DataFrame(rows).set_index("class")


def up_report(y_true_cls, y_pred_cls):
    """the false positive view: the model said up, the day did not go up"""
    said_up = np.asarray(y_pred_cls) == "up"
    went_up = np.asarray(y_true_cls) == "up"

    tp = int((said_up & went_up).sum())
    fp = int((said_up & ~went_up).sum())
    fn = int((~said_up & went_up).sum())

    return {
        "up_calls": tp + fp,
        "true_positives": tp,
        "false_positives": fp,
        "missed_ups": fn,
        "precision": tp / (tp + fp) if (tp + fp) else np.nan,
        "recall": tp / (tp + fn) if (tp + fn) else np.nan,
    }

## 8. Plots

Two diagnostics: the predicted line against the actual one over the test month with the
tie band drawn in, and the confusion matrix.

In [ ]:
# tokens - one blue, one orange for the two series, a blue ramp for the matrix
INK = "#0b0b0b"
INK_SOFT = "#52514e"
SURFACE = "#fcfcfb"
GRID = "#dcdbd6"
ACTUAL = "#2a78d6"
PREDICTED = "#eb6834"
BLUE_RAMP = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    ax.figure.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_SOFT, labelsize=9)
    ax.grid(True, color=GRID, linewidth=0.8, alpha=0.6)
    ax.set_axisbelow(True)


def plot_prediction(dates, y_true, y_pred, stock, up=UP_THRESHOLD, down=DOWN_THRESHOLD):
    fig, ax = plt.subplots(figsize=(11, 4.2))
    style_axes(ax)

    # the tie band, drawn first so the lines sit on top of it
    ax.axhspan(down, up, color=GRID, alpha=0.45, linewidth=0, zorder=0)
    ax.axhline(0, color=GRID, linewidth=1, zorder=1)

    ax.plot(dates, y_true, color=ACTUAL, linewidth=2, marker="o", markersize=5,
            label="actual", zorder=3)
    ax.plot(dates, y_pred, color=PREDICTED, linewidth=2, marker="o", markersize=5,
            label="predicted", zorder=3)

    ax.annotate("tie band", xy=(dates[0], up), xytext=(4, 4),
                textcoords="offset points", color=INK_SOFT, fontsize=9)

    ax.set_title(f"{stock} - next day close-open %, test month", color=INK,
                 fontsize=13, pad=34, loc="left")
    ax.set_ylabel("% of open", color=INK_SOFT, fontsize=10)
    # above the plot, clear of the rotated date labels below it
    legend = ax.legend(frameon=False, loc="lower left", fontsize=10,
                       bbox_to_anchor=(0, 1.01), ncol=2)
    for text in legend.get_texts():
        text.set_color(INK_SOFT)
    fig.autofmt_xdate(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_confusion(cm, stock, classes=CLASSES):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    ax.figure.set_facecolor(SURFACE)

    ax.imshow(cm, cmap=BLUE_RAMP, vmin=0, vmax=max(cm.max(), 1))

    ax.set_xticks(range(len(classes)), classes)
    ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("predicted", color=INK_SOFT, fontsize=10)
    ax.set_ylabel("actual", color=INK_SOFT, fontsize=10)
    ax.tick_params(colors=INK_SOFT, labelsize=10, length=0)
    for side in ax.spines.values():
        side.set_visible(False)

    # count in every cell, light ink once the fill goes dark
    limit = max(cm.max(), 1) * 0.6
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=12,
                    color="#ffffff" if cm[i, j] > limit else INK)

    ax.set_title(f"{stock} - confusion matrix", color=INK, fontsize=13, pad=12, loc="left")
    plt.tight_layout()
    plt.show()

## 9. Run one stock

Everything above, wired together. `run_stock` is what the multi stock cell calls too.

In [ ]:
def run_stock(stock, plots=True, verbose=True):
    """train and score one stock, returns a dict of everything worth keeping"""
    seed_everything()

    df = load_prices(csv_path(stock))
    year = pick_year(df)
    df = df[df["year"] == year].reset_index(drop=True)
    df = add_volume_norm(df)

    X_train, y_train, _ = make_windows(df, TRAIN_MONTHS)
    X_test, y_test, test_dates = make_windows(df, TEST_MONTHS)

    if verbose:
        print(f"\n=== {EXCHANGE}:{stock}  {year} ===")
        print(f"train months {TRAIN_MONTHS}: {X_train.shape[0]} windows")
        print(f"test  months {TEST_MONTHS}: {X_test.shape[0]} windows")
        truth = classify(y_test)
        counts = pd.Series(truth).value_counts().reindex(CLASSES, fill_value=0)
        print(f"actual test classes: {counts.to_dict()}")

    model = build_model()
    history = train_model(model, X_train, y_train)

    y_pred = model.predict(X_test, verbose=0).ravel()

    # RMSE, next to the two baselines it has to beat to mean anything
    scores = {
        "rmse": rmse(y_test, y_pred),
        "rmse_zero": rmse(y_test, np.zeros_like(y_test)),
        "rmse_train_mean": rmse(y_test, np.full_like(y_test, y_train.mean())),
    }

    true_cls = classify(y_test)
    pred_cls = classify(y_pred, up=UP_DECISION, down=DOWN_DECISION)
    cm = confusion(true_cls, pred_cls)
    up = up_report(true_cls, pred_cls)

    if verbose:
        print(f"\ntrain loss {history.history['loss'][-1]:.4f}")
        print(f"RMSE            {scores['rmse']:.3f}")
        print(f"  baseline 0%   {scores['rmse_zero']:.3f}")
        print(f"  baseline mean {scores['rmse_train_mean']:.3f}")
        print("\nconfusion matrix (rows actual, cols predicted)")
        print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))
        print("\nper class")
        print(per_class_scores(cm).round(3))
        print("\nup calls")
        for k, v in up.items():
            print(f"  {k:<16} {v:.3f}" if isinstance(v, float) else f"  {k:<16} {v}")
        if up["up_calls"] == 0:
            print("\n  the model never called an up day, so precision is undefined."
                  "\n  See the threshold sweep below.")

    if plots:
        plot_prediction(test_dates, y_test, y_pred, stock)
        plot_confusion(cm, stock)

    return {
        "stock": stock, "year": year, "model": model, "history": history,
        "dates": test_dates, "y_true": y_test, "y_pred": y_pred,
        "cm": cm, "up": up, **scores,
    }


result = run_stock(STOCK)

## 10. Trading precision for recall

An MSE-trained regressor on ~17 rows predicts close to the mean, so it often never
crosses +1.5% and calls no up days at all. This sweeps the decision threshold: each row
is what would happen if the model called "up" above that predicted percentage.

Read it as the precision/volume trade. A threshold with high precision and two up calls
in a month is not evidence of anything - the `up_calls` column is there to stop a
one-out-of-one from reading as 100%.

In [ ]:
def threshold_sweep(y_true, y_pred, thresholds=None):
    """precision of 'up' as the decision threshold moves"""
    if thresholds is None:
        thresholds = np.round(np.arange(-1.0, 2.01, 0.25), 2)

    true_cls = classify(y_true)
    rows = []
    for t in thresholds:
        pred_cls = np.where(y_pred >= t, "up", "tie")
        stats = up_report(true_cls, pred_cls)
        rows.append({"threshold": t, **stats})

    return pd.DataFrame(rows).set_index("threshold").round(3)


sweep = threshold_sweep(result["y_true"], result["y_pred"])
print(f"actual up days in the test month: {int((classify(result['y_true']) == 'up').sum())}"
      f" of {len(result['y_true'])}\n")
print(sweep)

## 11. Every stock

One model per stock, same pipeline, one row of results each. Edit `STOCKS` to match the
csv files you actually have - a stock whose csv is missing is reported and skipped.

In [ ]:
STOCKS = [STOCK]   # e.g. ["COMI", "HRHO", "TMGH", "SWDY", "NIPH"]

def run_all(stocks=STOCKS):
    rows, failed = [], []

    for stock in stocks:
        try:
            r = run_stock(stock, plots=False, verbose=False)
        except (FileNotFoundError, ValueError) as e:
            print(f"{stock}: skipped - {e}")
            failed.append(stock)
            continue

        rows.append({
            "stock": stock,
            "year": r["year"],
            "rmse": round(r["rmse"], 3),
            "rmse_zero": round(r["rmse_zero"], 3),
            "beats_baseline": r["rmse"] < r["rmse_zero"],
            "up_calls": r["up"]["up_calls"],
            "true_pos": r["up"]["true_positives"],
            "false_pos": r["up"]["false_positives"],
            "up_precision": round(r["up"]["precision"], 3)
                            if r["up"]["up_calls"] else np.nan,
        })
        print(f"{stock}: done")

    if failed:
        print(f"\nskipped: {', '.join(failed)}")

    return pd.DataFrame(rows).set_index("stock") if rows else pd.DataFrame()


summary = run_all()
summary